In [2]:
from Bio import SeqIO
from Bio.PDB import PDBParser, PPBuilder
import os

def extract_seq_from_pdb(pdb_file):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("vaccine", pdb_file)
    ppb = PPBuilder()
    sequence = ""
    for pp in ppb.build_peptides(structure):
        sequence += str(pp.get_sequence())
    return sequence

def load_fasta_sequences(fasta_file):
    return list(SeqIO.parse(fasta_file, "fasta"))

def compare_sequences(seq1, seq2):
    diffs = []
    for i, (a, b) in enumerate(zip(seq1, seq2), start=1):
        if a != b:
            diffs.append((i, a, b))
    return diffs

# === INPUT PATHS ===
pdb_folder = "pdbs"  # Folder with vaccine1.pdb to vaccine16.pdb
fasta_main = "vaccines_with_tpa_adjuvant.fasta"
fasta_compare = "vaccines_with_adjuvants.fasta"

# === PART 1: Match PDB to FASTA ===
fasta_seqs = load_fasta_sequences(fasta_main)

print("=== Verifying PDB files against FASTA ===")
for i, fasta_record in enumerate(fasta_seqs, start=1):
    pdb_file = os.path.join(pdb_folder, f"vaccine{i}.pdb")
    if not os.path.exists(pdb_file):
        print(f"[!] Missing: {pdb_file}")
        continue
    pdb_seq = extract_seq_from_pdb(pdb_file)
    fasta_seq = str(fasta_record.seq)

    if pdb_seq == fasta_seq:
        print(f"[✓] vaccine{i} - MATCHED")
    else:
        print(f"[✗] vaccine{i} - MISMATCH (Length {len(pdb_seq)} vs {len(fasta_seq)})")
        diffs = compare_sequences(pdb_seq, fasta_seq)
        print(f"    → {len(diffs)} differences")
        if diffs:
            print(f"    First few: {diffs[:5]}")

# === PART 2: Compare Two FASTA Files ===
print("\n=== Comparing FASTA: vaccines_with_adjuvants vs vaccines_with_tpa_adjuvant ===")
fasta_tpa = load_fasta_sequences(fasta_main)
fasta_plain = load_fasta_sequences(fasta_compare)

for i, (rec1, rec2) in enumerate(zip(fasta_plain, fasta_tpa), start=1):
    diffs = compare_sequences(str(rec1.seq), str(rec2.seq))
    if diffs:
        print(f"[✗] Vaccine {i}: {len(diffs)} difference(s)")
        for pos, a, b in diffs[:5]:
            print(f"    Pos {pos}: {a} → {b}")
    else:
        print(f"[✓] Vaccine {i}: Identical")



=== Verifying PDB files against FASTA ===
[✗] vaccine1 - MISMATCH (Length 310 vs 331)
    → 255 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine2 - MISMATCH (Length 310 vs 331)
    → 257 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine3 - MISMATCH (Length 310 vs 331)
    → 256 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine4 - MISMATCH (Length 310 vs 331)
    → 254 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine5 - MISMATCH (Length 310 vs 331)
    → 253 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine6 - MISMATCH (Length 310 vs 331)
    → 253 differences
    First few: [(1, 'A', 'M'), (2, 'P', 'D'), (3, 'P', 'A'), (4, 'H', 'M'), (5, 'A', 'K')]
[✗] vaccine7 - M

In [16]:
from Bio import SeqIO
from Bio.PDB import PDBParser, PPBuilder
import os

def load_fasta_sequences(filepath):
    sequences = {}
    for record in SeqIO.parse(filepath, "fasta"):
        # Normalize headers like "Vaccine 4:" → "vaccine4"
        norm_id = record.id.lower().replace(" ", "").replace(":", "")
        sequences[norm_id] = str(record.seq)
    return sequences

def extract_sequence_from_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    ppb = PPBuilder()
    try:
        structure = parser.get_structure("structure", pdb_path)
        sequence = ""
        for model in structure:
            for chain in model:
                for pp in ppb.build_peptides(chain):
                    sequence += str(pp.get_sequence())
        return sequence
    except Exception as e:
        print(f"[!] Error reading {pdb_path}: {e}")
        return None

# === File paths ===
pdb_folder = "pdbs"
fasta1_path = "vaccines_with_adjuvants.fasta"
fasta2_path = "vaccines_with_tpa_adjuvant.fasta"

# === Load FASTA sequences ===
fasta1_seqs = load_fasta_sequences(fasta1_path)
fasta2_seqs = load_fasta_sequences(fasta2_path)

# === Compare PDBs ===
for i in range(1, 17):
    name = f"vaccine{i}"
    pdb_path = os.path.join(pdb_folder, f"{name}.pdb")

    print(f"\n--- {name} ---")
    if not os.path.exists(pdb_path):
        print("  [!] PDB file missing.")
        continue

    pdb_seq = extract_sequence_from_pdb(pdb_path)
    if not pdb_seq:
        print("  [!] Could not extract sequence from PDB.")
        continue

    # Check against FASTA 1
    seq1 = fasta1_seqs.get(name)
    if not seq1:
        print("  [!] Missing in FASTA 1")
    else:
        mismatches1 = [(i + 1, a, b) for i, (a, b) in enumerate(zip(pdb_seq, seq1)) if a != b]
        length_diff1 = abs(len(pdb_seq) - len(seq1))
        if mismatches1 or length_diff1:
            print(f"  [✗] Mismatch with FASTA 1 ({len(mismatches1)} differences, length diff {length_diff1})")
            print("      First 5 mismatches:", mismatches1[:5])
        else:
            print("  [✓] Matches FASTA 1")

    # Check against FASTA 2
    seq2 = fasta2_seqs.get(name)
    if not seq2:
        print("  [!] Missing in FASTA 2")
    else:
        mismatches2 = [(i + 1, a, b) for i, (a, b) in enumerate(zip(pdb_seq, seq2)) if a != b]
        length_diff2 = abs(len(pdb_seq) - len(seq2))
        if mismatches2 or length_diff2:
            print(f"  [✗] Mismatch with FASTA 2 ({len(mismatches2)} differences, length diff {length_diff2})")
            print("      First 5 mismatches:", mismatches2[:5])
        else:
            print("  [✓] Matches FASTA 2")



--- vaccine1 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine2 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine3 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine4 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine5 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine6 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine7 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine8 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine9 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine10 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine11 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine12 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine13 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine14 ---
  [!] Missing in FASTA 1
  [!] Missing in FASTA 2

--- vaccine15 ---
  [!] Missing in FASTA 1

In [17]:
import os
from Bio.PDB import PDBParser, Polypeptide

def extract_seq_from_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('', pdb_path)
    seq = ''
    for model in structure:
        for chain in model:
            # extract amino acid sequence from each chain
            ppb = Polypeptide.PPBuilder()
            for pp in ppb.build_peptides(chain):
                seq += str(pp.get_sequence())
    return seq

# The two folders
folders = ['adjuvants', 'noadjuvants']

for folder in folders:
    fasta_lines = []
    for file in os.listdir(folder):
        if file.lower().endswith('.pdb'):
            pdb_path = os.path.join(folder, file)
            header = os.path.splitext(file)[0]  # remove .pdb
            sequence = extract_seq_from_pdb(pdb_path)
            fasta_lines.append(f'>{header}\n{sequence}')
    
    # write a single fasta file per folder
    fasta_path = f'{folder}.fasta'
    with open(fasta_path, 'w') as f:
        f.write('\n'.join(fasta_lines))
    print(f'Wrote {fasta_path} containing {len(fasta_lines)} sequences.')


Wrote adjuvants.fasta containing 16 sequences.
Wrote noadjuvants.fasta containing 16 sequences.


In [18]:
from Bio import SeqIO

# New FASTA files
new_fastas = ['adjuvants.fasta', 'noadjuvants.fasta']

# Old/reference FASTA files
old_fastas = ['vaccines_with_adjuvants.fasta', 'vaccines_with_tpa_adjuvant.fasta']

# Load old sequences into a dictionary: {filename: {seq_string: [labels]}}
old_db = {}

for old_file in old_fastas:
    seq_dict = {}
    for record in SeqIO.parse(old_file, 'fasta'):
        seq = str(record.seq)
        label = record.id
        seq_dict.setdefault(seq, []).append(label)
    old_db[old_file] = seq_dict

# Compare new sequences to all old ones
for new_file in new_fastas:
    print(f"\n=== Checking {new_file} ===")
    for new_record in SeqIO.parse(new_file, 'fasta'):
        new_seq = str(new_record.seq)
        new_label = new_record.id

        for old_file, seq_dict in old_db.items():
            if new_seq in seq_dict:
                for matching_old_label in seq_dict[new_seq]:
                    print(f"> Match found: NEW ({new_file} :: {new_label})  <==>  OLD ({old_file} :: {matching_old_label})")



=== Checking adjuvants.fasta ===
> Match found: NEW (adjuvants.fasta :: Vaccine1)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine1:)
> Match found: NEW (adjuvants.fasta :: Vaccine10)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine10:)
> Match found: NEW (adjuvants.fasta :: Vaccine11)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine11:)
> Match found: NEW (adjuvants.fasta :: Vaccine12)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine12:)
> Match found: NEW (adjuvants.fasta :: Vaccine13)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine13:)
> Match found: NEW (adjuvants.fasta :: Vaccine14)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine14:)
> Match found: NEW (adjuvants.fasta :: Vaccine15)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine15:)
> Match found: NEW (adjuvants.fasta :: Vaccine16)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine16:)
> Match found: NEW (adjuvants.fasta :: Vaccine2)  <==>  OLD (vaccines_with_adjuvants.fasta :: Vaccine2:)
> Match